In [ ]:
pip install plotly panel

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import panel as pn
import plotly.graph_objects as go

In [12]:
def load_data():

    # Read transactions data from CSV
    TransDF = pd.read_csv('categorised_transactions.csv', index_col=False)
    # Remove any unnamed index columns from the CSV
    TransDF = TransDF.loc[:, ~TransDF.columns.astype(str).str.startswith('Unnamed')]
    # Add year and month columns
    TransDF['Year'] = pd.to_datetime(TransDF['Date']).dt.year
    TransDF['Month'] = pd.to_datetime(TransDF['Date']).dt.month
    TransDF['Month Name'] = pd.to_datetime(TransDF['Date']).dt.strftime("%b")
    # Add FY column
    TransDF['FYNumber'] = np.where(TransDF['Month'] >= 7, TransDF['Year'], TransDF['Year'] - 1)
    # Format as "FYYYY/YY" (e.g., FY2024/25)
    TransDF['FY'] = (
        'FY' + TransDF['FYNumber'].astype(str) + '/' + 
        (TransDF['FYNumber'] + 1).astype(str).str[-2:]
        )

    TransDF['Income/Expense'] = np.where(TransDF['Amount']>0, 'Income','Expense')
    
    return TransDF

TransDF = load_data()
TransDF



C:\Users\plum\AppData\Local\Temp\ipykernel_31348\624238353.py:8: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  TransDF['Year'] = pd.to_datetime(TransDF['Date']).dt.year
C:\Users\plum\AppData\Local\Temp\ipykernel_31348\624238353.py:9: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  TransDF['Month'] = pd.to_datetime(TransDF['Date']).dt.month
C:\Users\plum\AppData\Local\Temp\ipykernel_31348\624238353.py:10: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  TransDF['Month Name'] = pd.to_datetime(TransDF['Date']).dt.strftime("%b")


,Date,Amount,Transaction,Category,Year,Month,Month Name,FYNumber,FY,Income/Expense
0,27/03/2025,-288,Yarravalleywater Yvow,Utility,2025,3,Mar,2024,FY2024/25,Expense
1,11/06/2025,-40,Target Westfield,Retails,2025,6,Jun,2024,FY2024/25,Expense
2,02/10/2025,-197,Barwon Water,Utility,2025,10,Oct,2025,FY2025/26,Expense
3,22/07/2024,-1500,Virgin Airlines,Travel & Transport,2024,7,Jul,2024,FY2024/25,Expense
4,27/01/2026,-21,Woolworths Great Western,Supplies & Essentials,2026,1,Jan,2025,FY2025/26,Expense
5,15/01/2025,-62,Service Nsw,Government,2025,1,Jan,2024,FY2024/25,Expense
6,19/06/2026,6372,From Metro Art Galary,Dining & Leisure,2026,6,Jun,2025,FY2025/26,Income
7,11/11/2024,167,Yellow Bird Painting,Creative,2024,11,Nov,2024,FY2024/25,Income
8,01/12/2025,-33,Aldi Mobile,Retails,2025,12,Dec,2025,FY2025/26,Expense
9,04/11/2024,2067,Transfer From Macquarie University,Education,2024,11,Nov,2024,FY2024/25,Income


In [13]:
def create_pie_chart(TransDF, NOrder):

    color_scale = px.colors.qualitative.Set1

    CurFY = sorted(TransDF['FY'].dropna().unique())[-NOrder]
    CurFYDf = TransDF[TransDF['FY'].eq(CurFY)].copy()
    CurFYDf['Amount'] = pd.to_numeric(CurFYDf['Amount'], errors='coerce').abs()

    CurFYSummary = (
        CurFYDf.groupby('Income/Expense', as_index=False)['Amount']
        .sum()
    )

    IncomeValue = CurFYSummary.loc[CurFYSummary['Income/Expense'] == 'Income', 'Amount'].sum()
    ExpenseValue = CurFYSummary.loc[CurFYSummary['Income/Expense'] == 'Expense', 'Amount'].sum()

    CurFYSummary['SharePercentOfIncome'] = CurFYSummary['Amount'] / IncomeValue * 100 if IncomeValue > 0 else 0
    SavingsPercent = (IncomeValue - ExpenseValue) / IncomeValue * 100 if IncomeValue else 0

    CurFYSummary['DisplayText'] = CurFYSummary.apply(
        lambda row: (
            f"Savings<br>{SavingsPercent:.1f}%"
            if row['Income/Expense'] == 'Income'
            else f"Expense<br>{row['SharePercentOfIncome']:.1f}%"
        ),
        axis=1
    )
    #plot pie for % metrics
    FigPie = px.pie(
        CurFYSummary,
        # names='LegendLabel',
        values='Amount',
        title=f'Financial Year ({CurFY}) Share by Savings/Expense (%)',
        hole=0.3,
        color_discrete_sequence=color_scale
    )
    #label (Expense, Savings) the metrics in segments
    FigPie.update_traces(
        text=CurFYSummary['DisplayText'],
        textinfo='text',
        textposition='inside',
        hoverinfo='skip',
        hovertemplate=None,
        marker=dict(line=dict(color='white', width=1))
    )
    #label the center of pie chart
    FigPie.add_annotation(        
        text=f'Income<br>{IncomeValue / IncomeValue * 100:,.0f}%',
        x=0.5,
        y=0.5,
        showarrow=False,
        xref='paper',
        yref='paper',
        align='center',
        font=dict(size=16, color='black')
    )
    # Control the legend location
    FigPie.update_layout(
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=-0.2,
            xanchor='center',
            x=0.5,
            title_text=''
        ),
        title=dict(x=0.5)
    )
    return FigPie


In [14]:
PieChart1 = create_pie_chart(TransDF, 1)
PieChart1.show()
PieChart2 = create_pie_chart(TransDF, 2)
PieChart2.show()

In [156]:
def create_year_comparison_bar_chart(TransDF):
    FYValues = sorted(TransDF["FY"].dropna().unique())[-2:]
    ChartRows = []

    for FY in FYValues:
        FYDF = TransDF[TransDF["FY"].eq(FY)].copy()
        FYDF["Amount"] = pd.to_numeric(FYDF["Amount"], errors="coerce")

        IncomeValue = FYDF.loc[FYDF["Amount"] > 0, "Amount"].sum()
        ExpenseValue = FYDF.loc[FYDF["Amount"] < 0, "Amount"].abs().sum()

        ExpensePercent = (ExpenseValue / IncomeValue * 100) if IncomeValue > 0 else 0
        
        SavingsPercent = ((IncomeValue - ExpenseValue) / IncomeValue * 100) if IncomeValue > 0 else 0 
        
        ChartRows.append({"FY": FY, "Performance": "Expense %", "Percent": ExpensePercent})
        
        ChartRows.append({"FY": FY, "Performance": "Savings %", "Percent": SavingsPercent})
        
    ComparisonDF = pd.DataFrame(ChartRows)

    BarFig = px.bar(
        ComparisonDF,
        x="FY",
        y="Percent",
        color="Performance",
        barmode="group",
        title="Expense vs Savings Percentage across Two Financial Years",
        labels={"Percent": "", "FY": ""},
        # labels={"Percent": "Percentage (%)", "FY": "Financial Year"},
    )

    BarFig.update_traces(
        texttemplate="%{y:.1f}%", 
        textposition="outside",
        hoverinfo='skip',
        hovertemplate=None)
    
    BarFig.update_layout(
        legend_title="", 
        yaxis=dict(range=[0, 110], showticklabels=False),
        height=350,
        # width=550,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        legend=dict(
            orientation="h",    
            yanchor="top",      
            y=-0.2,             
            xanchor="center",   
            x=0.5               
        ),
        title=dict(
            x=0.5,             
            xanchor='center'   
        )
            
    )

    return BarFig, ComparisonDF 

ComparisonChart, ComparisonDF = create_year_comparison_bar_chart(TransDF)
ComparisonDF

,FY,Performance,Percent
0,FY2024/25,Expense %,76.693581
1,FY2024/25,Savings %,23.306419
2,FY2025/26,Expense %,89.077213
3,FY2025/26,Savings %,10.922787


In [17]:
def FormatVariance(Value):
    Suffix = " (Increase)" if Value > 0 else " (Decrease)" if Value < 0 else ""
    return f"{Value:+.2f}%{Suffix}"

In [ ]:
PivotDF = ComparisonDF.pivot(
    index="Performance", columns="FY", values="Percent"
)

BaselinePeriod = PivotDF.columns[0]  
CurrentPeriod = PivotDF.columns[1]  

# Initialize the summary structure
SummaryDF = pd.DataFrame(index=PivotDF.index)
SummaryDF[BaselinePeriod] = PivotDF[BaselinePeriod]
SummaryDF[CurrentPeriod] = PivotDF[CurrentPeriod]

SummaryDF["Net Percentage Point Change"] = (
    PivotDF[CurrentPeriod] - PivotDF[BaselinePeriod]
)

# Disable row heading
SummaryDF.index.name = None
SummaryDF = SummaryDF.round(1)

SummaryDF[BaselinePeriod] = SummaryDF[BaselinePeriod].astype(str) + "%"
SummaryDF[CurrentPeriod] = SummaryDF[CurrentPeriod].astype(str) + "%"
SummaryDF["Net Percentage Point Change"] = (
    SummaryDF["Net Percentage Point Change"].map(FormatVariance)
)

# Create  a simple data list 
TableData = [
    SummaryDF.index,
    SummaryDF[BaselinePeriod],
    SummaryDF[CurrentPeriod],
    SummaryDF["Net Percentage Point Change"],
]

# 3. Build the clean table using direct color strings
StyledTable = go.Figure(
    data=go.Table(
        header=dict(
            values=["Performance", BaselinePeriod, CurrentPeriod, "Net Change"],
            font=dict(size=14, color="#064E3B"), 
            fill_color="#FEB408",  
            line_color="#F3B310", 
            line_width=1
        ),
        cells=dict(
            values=TableData,
            font=dict(size=13, color="#1E293B"),  
            fill_color="#FBD756",  
            line_color="#F3B310",  
            line_width=1
        ),
    )
)

# formating layout
StyledTable.update_layout(
    height=200,  # Increased from 160 to make room for the title text
    width= 440,
    paper_bgcolor="#FFFFFF",
    plot_bgcolor="#F59E9E",
    margin=dict(
        l=10, r=10, t=40, b=10
    ),  # Increased top margin (t) from 10 to 50
    title=dict(
        text="<span style='font-weight:normal; font-size:16px; color:#1E293B;'> Comparative Summary for Financial Years</span>",
        x=0.5,
        y=0.95
    ),
)
    
StyledTable.show()


In [166]:
def create_bar_chart(TransDF, DateOrder):

    color_scale = px.colors.qualitative.Set1

    #sort the data on year and pick out the year requested
    CurFY = sorted(TransDF['FY'].dropna().unique())[-DateOrder]
    CurYearDF = TransDF[TransDF['FY'].eq(CurFY)].copy()
    CurYearDF['Amount'] = pd.to_numeric(CurYearDF['Amount'], errors='coerce').abs()
    CurYearDF['Year - Month'] = CurYearDF['Year'].astype(str) + ' - ' + CurYearDF['Month Name']
                                                                                  
    # take only expense trans
    TransDFExpensesOnly = CurYearDF[(CurYearDF['Income/Expense'] == 'Expense')]
    
   
    TransDFGrouped = (
        TransDFExpensesOnly.groupby(["Month", "Year", "Category", "Month Name", 'Year - Month'])["Amount"]
        .sum()
        .reset_index()
    )
    
    # #Calculate the monthly totals across all categories per month      
    MonthlyTotals = TransDFGrouped.groupby('Year - Month')['Amount'].transform('sum')
    
    TransDFGrouped['Percent %'] = (TransDFGrouped['Amount'] / MonthlyTotals) * 100

    TransDFGrouped = TransDFGrouped.sort_values(by=["Year", "Month"])

    SortedMonthNames = TransDFGrouped['Year - Month'].unique().tolist()
   
    # Build the stacked chart from here 
    BarChartFigure = px.bar(
        TransDFGrouped,
        x='Year - Month',
        y="Percent %",
        color="Category", 
        title=f"Monthly Expense Breakdown by Category Summary ({CurFY})",
        labels={"Percent %": "", "Year - Month": ""},
        text_auto=True,
        color_discrete_sequence=color_scale,
        category_orders={'Year - Month': SortedMonthNames} 
    )
   
    BarChartFigure.update_traces(
        texttemplate='%{y:.0f}%', 
        textposition='inside',
        hoverinfo='skip',
        hovertemplate=None
    )
    
    BarChartFigure.update_layout(
        yaxis=dict(showticklabels=False),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        barmode="stack", 
        height=400,
        width=660,
        legend=dict(
                    orientation="h",    
                    yanchor="top",      
                    y=-0.3,             
                    xanchor="center",   
                    x=0.5               
                ),
        title=dict(
            x=0.5,             
            xanchor='center',
            y=0.85  
        )
    )

    return BarChartFigure

FY1Category = create_bar_chart(TransDF, 2)
FY2Category = create_bar_chart(TransDF, 1)
FY1Category.show()
FY2Category.show()


In [167]:
# Create tabs
TableWrapper = pn.Column(StyledTable, margin=(100, 10, 10, 0))
ChartWrapper = pn.Column(ComparisonChart, margin=(50, 10, 10, 10))
Chart1Wrapper = pn.Column(FY1Category, margin=(10, 0, 0, 0))
Chart2Wrapper = pn.Column(FY2Category, margin=(10, 0, 0, 0))

tabs = pn.Tabs(
                        ('FY Performance Summary', pn.Column(pn.Row(ChartWrapper,TableWrapper))),
                        #                         pn.Row(expense_category_monthly_FY25, expense_category_monthly_FY26))),
                        ('Monthly FY Performance Breakdown by Category Summary', pn.Column(pn.Row(Chart1Wrapper, Chart2Wrapper)))
                        #                         pn.Row(income_monthly_2023, expense_monthly_2023))
                        
                )
tabs.show()

Launching server at http://localhost:64068


In [169]:
# Dashboard template
template = pn.template.FastListTemplate(
    title='Banking Statement Insights Dashboard',
    sidebar=[pn.pane.Markdown("Income & Expense Insights"), 
             pn.pane.Markdown("The dashboard provides a high level analysis of income, " \
             "expenses, and savings margins using synthetic banking data. Its designed to " \
             "give a quick understanding of financial health trends across multiple financial years." \
             "The dashboard uses synthetic transaction data and categorized using a local LLM, ensuring privacy " \
             "while still enabling meaningful insights."),
             pn.pane.PNG("./images/insights.png", sizing_mode="scale_both")
             ],
    main=[pn.Row(pn.Column(pn.Row(tabs))),],
    #accent_base_color="#88d8b0",
    header_background="#c0b9dd",
)

template.show()

Launching server at http://localhost:64559
